**Name:** Syed Obaidullah Hassan Chishti  
**Roll No:** 25280049

# Imports

In [1]:
import psycopg2
from dotenv import load_dotenv
import os
import pandas as pd
import great_expectations as gx
from typing import Any, Dict, List
import json
from pathlib import Path
import numpy as np

# Load environment variables from .env file
load_dotenv(override=True)

True

# Global Variables

In [2]:
DATABASE_NAME = "de_assignment3"  
CLEAN_DATA_TABLE_NAME = "clean_data"
CORRUPTED_DATA_TABLE_NAME = "corrupted_data"
CLEAN_DATA_CSV_PATH = "25280049_clean_synthetic_dataset.csv"
CORRUPTED_DATA_CSV_PATH = "25280049_corrupted_synthetic_dataset.csv"

# Data Quality and Validation in Real-World Datasets

## Task 1: Generate a Large Synthetic Dataset with Real-World Corruptions

### Connect to local PostgreSQL

In [3]:
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"), 
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT")  
)

conn.autocommit = True  

### Create Database

In [4]:
cur = conn.cursor()

# Check if the database already exists
cur.execute(
    "SELECT 1 FROM pg_database WHERE datname = %s;",
    (DATABASE_NAME,)
)
exists = cur.fetchone()

if not exists:
    # Create the database if it does not exist
    cur.execute(f"CREATE DATABASE {DATABASE_NAME};")
    print(f"Database {DATABASE_NAME} created!")
else:
    print(f"Database {DATABASE_NAME} already exists.")

cur.close()
conn.close()

Database de_assignment3 already exists.


In [5]:
# Reconnect to PostgreSQL but this time using the database we just created
conn = psycopg2.connect(
    host=os.getenv("POSTGRES_HOST"), 
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    port=os.getenv("POSTGRES_PORT"),
    dbname=DATABASE_NAME
)

conn.autocommit = True
print(f"Connected to database {DATABASE_NAME}")


Connected to database de_assignment3


### Drop Old Tables if Exist

In [6]:
cur = conn.cursor()
cur.execute(f"DROP TABLE IF EXISTS {CLEAN_DATA_TABLE_NAME};")
cur.execute(f"DROP TABLE IF EXISTS {CORRUPTED_DATA_TABLE_NAME};")
cur.close()

### Create Tables

In [7]:
cur = conn.cursor()

cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {CLEAN_DATA_TABLE_NAME} 
                (sale_id BIGSERIAL PRIMARY KEY,
                customer_id INT NOT NULL,
                product_id INT NOT NULL, 
                sale_ts TIMESTAMP NOT NULL,
                quantity INT NOT NULL,
                amount NUMERIC(12, 2) NOT NULL,
                channel TEXT NOT NULL,
                currency TEXT NOT NULL,
                notes TEXT);
            """)

cur.execute(f"""
                CREATE TABLE IF NOT EXISTS {CORRUPTED_DATA_TABLE_NAME} 
                (sale_id BIGINT, 
                customer_id TEXT,
                product_id TEXT,
                sale_ts TIMESTAMP,
                quantity INT,
                amount NUMERIC(12, 2),
                channel TEXT,
                currency TEXT,
                notes TEXT);
            """)

cur.close()

print("Tables created successfully!")

Tables created successfully!


### Insert data in Clean Table

In [8]:
CLEAN_DATA_SQL = f"""
DO $$
DECLARE
    N_BASE BIGINT := 1000000;
BEGIN
    INSERT INTO {CLEAN_DATA_TABLE_NAME} (customer_id, product_id, sale_ts, quantity, amount,
        channel, currency, notes)
    SELECT
        CASE
            WHEN random() < 0.05 THEN (1 + floor(random()*5000))::int
            ELSE (1 + floor(random()*200000))::int
        END AS customer_id,
        CASE
            WHEN random() < 0.10 THEN (1 + floor(random()*200))::int
            ELSE (1 + floor(random()*20000))::int
        END AS product_id,
        (now() - (floor(random()*365))::int * interval '1 day'
               - (floor(random()*24))::int * interval '1 hour'
               - (floor(random()*60))::int * interval '1 minute'
        ) AS sale_ts,
        CASE
            WHEN random() < 0.90 THEN (1 + floor(random()*4))::int
            WHEN random() < 0.99 THEN (5 + floor(random()*10))::int
            ELSE (20 + floor(random()*200))::int
        END AS quantity,
        ROUND(
            ((10 + random()*200)
            * (CASE WHEN random() < 0.90 THEN 1 ELSE (1 + random()*3) END)
            * (CASE WHEN random() < 0.85 THEN 1 ELSE (0.2 + random()*0.8) END)
            )::numeric
            * (CASE WHEN random() < 0.98 THEN 1 ELSE (10 + random()*50) END)::numeric
        , 2) AS amount,
        (ARRAY['online','store','partner','mobile_app'])[1 + floor(random()*4)] AS channel,
        (ARRAY['USD','EUR','GBP','PKR'])[1 + floor(random()*4)] AS currency,
        NULL::text AS notes
    FROM generate_series(1, N_BASE);
END $$;
"""

In [9]:
curr = conn.cursor()
curr.execute(CLEAN_DATA_SQL)
curr.close()

### Insert data in Corrupted Table

In [10]:
# Copy Exact Clean Data First to Corrupted Data Table

COPY_CLEAN_DATA_TO_CORRUPTED_DATA_SQL = f"""
INSERT INTO {CORRUPTED_DATA_TABLE_NAME} 
(sale_id, customer_id, product_id, sale_ts, quantity, amount, channel, currency, notes)
SELECT 
sale_id, customer_id, product_id, sale_ts, quantity, amount, channel, currency, notes
FROM {CLEAN_DATA_TABLE_NAME}; 
"""

curr = conn.cursor()
curr.execute(COPY_CLEAN_DATA_TO_CORRUPTED_DATA_SQL) 
curr.close()

In [11]:
# Introduce Exact Duplicates

DUPLICATE_DATA_SQL = f"""
INSERT INTO {CORRUPTED_DATA_TABLE_NAME}
SELECT * FROM {CLEAN_DATA_TABLE_NAME}
WHERE random() < 0.02;
"""

curr = conn.cursor()
curr.execute(DUPLICATE_DATA_SQL)
curr.close()

In [12]:
# Introduce Near Duplicates

NEAR_DUPLICATE_DATA_SQL = f"""
INSERT INTO {CORRUPTED_DATA_TABLE_NAME}
(sale_id, customer_id, product_id, sale_ts, quantity, amount, channel, currency, notes)
SELECT
    sale_id,
    customer_id,
    product_id,
    sale_ts + (CASE WHEN random() < 0.5 THEN interval '0' ELSE interval '5 seconds' END),
    CASE WHEN random() < 0.8 THEN quantity ELSE quantity + 1 END,
    CASE WHEN random() < 0.7 THEN amount ELSE ROUND(amount + (random()*2 - 1)::numeric, 2) END,
    CASE
        WHEN random() < 0.33 THEN upper(channel)
        WHEN random() < 0.66 THEN ' ' || channel || ' '
        ELSE channel
    END,
    currency,
    'near-duplicate variant'
FROM {CLEAN_DATA_TABLE_NAME}
WHERE random() < 0.02;
"""

curr = conn.cursor()
curr.execute(NEAR_DUPLICATE_DATA_SQL)
curr.close()

In [13]:
# Inject Missing Values and Placeholder Tokens

MISSING_VALUES_SQL = f"""
UPDATE {CORRUPTED_DATA_TABLE_NAME}
SET
  customer_id = CASE WHEN random() < 0.25 THEN NULL ELSE customer_id END,
  product_id = CASE WHEN random() < 0.20 THEN NULL ELSE product_id END,
  sale_ts = CASE WHEN random() < 0.10 THEN NULL ELSE sale_ts END,
  channel = CASE
                WHEN random() < 0.15 THEN NULL
                WHEN random() < 0.30 THEN ''
                WHEN random() < 0.45 THEN 'N/A'
                WHEN random() < 0.60 THEN 'unknown'
                ELSE channel
            END,
  notes = CASE
              WHEN random() < 0.30 THEN 'missing fields injected'
              ELSE notes
          END
WHERE random() < 0.05;
"""

curr = conn.cursor()
curr.execute(MISSING_VALUES_SQL)
curr.close()

In [14]:
# Inject Invalid Ranges and Outliers

INVALID_RANGES_SQL = f"""
UPDATE {CORRUPTED_DATA_TABLE_NAME}
SET
    quantity = CASE
        WHEN random() < 0.25 THEN -1 * (1 + floor(random()*5))::int -- negative qty
        WHEN random() < 0.50 THEN 0 -- zero qty
        WHEN random() < 0.75 THEN (1000 + floor(random()*5000))::int -- extreme qty
        ELSE quantity
    END,
    amount = CASE
        WHEN random() < 0.25 THEN -1 * ROUND((random()*500)::numeric, 2) -- negative amount
        WHEN random() < 0.50 THEN 0::numeric(12,2) -- zero amount
        WHEN random() < 0.75 THEN ROUND((100000 + random()*900000)::numeric, 2) -- extreme amount
        ELSE amount
    END,
    notes = COALESCE(notes, '') || ' | invalid-range/outlier injected'
WHERE random() < 0.03;
"""

curr = conn.cursor()
curr.execute(INVALID_RANGES_SQL)
curr.close()

In [15]:
# Inject Timestamp Anomalies

TIMESTAMP_ANOMALIES_SQL = f"""
UPDATE {CORRUPTED_DATA_TABLE_NAME}
SET
    sale_ts = CASE
        WHEN random() < 0.50 THEN now() + (1 + floor(random()*30))::int * interval '1 day' -- future
        ELSE now() - (3650 + floor(random()*3650))::int * interval '1 day' -- 10-20y old
    END,
    notes = COALESCE(notes,'') || ' | timestamp anomaly'
WHERE random() < 0.02;
"""

curr = conn.cursor()
curr.execute(TIMESTAMP_ANOMALIES_SQL)
curr.close()

In [16]:
# Inject Referential Issues

REFERENTIAL_ISSUES_SQL = f"""
UPDATE {CORRUPTED_DATA_TABLE_NAME}
SET
    customer_id = CASE 
        WHEN random() < 0.5 THEN (300000 + floor(random()*500000))::int::text 
        ELSE customer_id 
    END,
    product_id = CASE 
        WHEN random() < 0.5 THEN (50000 + floor(random()*200000))::int::text 
        ELSE product_id 
    END,
    notes = COALESCE(notes,'') || ' | orphan ids'
WHERE random() < 0.02;
"""

curr = conn.cursor()
curr.execute(REFERENTIAL_ISSUES_SQL)
curr.close()

In [17]:
# Inject Currency Issues

CURRENCY_ISSUES_SQL = f"""
UPDATE {CORRUPTED_DATA_TABLE_NAME}
SET
    currency = CASE
        WHEN random() < 0.25 THEN '$' || amount::text 
        WHEN random() < 0.50 THEN amount::text || ' ' || currency 
        WHEN random() < 0.75 THEN replace(amount::text, '.', ',') 
        ELSE currency
    END,
    notes = COALESCE(notes, '') || ' | currency format issue'
WHERE random() < 0.04;
"""

curr = conn.cursor()
curr.execute(CURRENCY_ISSUES_SQL)
curr.close()

### Load Data from PostgreSQL and Store in CSV

In [18]:
# Store Clean Data

query = f"""
SELECT * FROM {CLEAN_DATA_TABLE_NAME};
"""

df_clean = pd.read_sql_query(query, conn)
df_clean.to_csv(CLEAN_DATA_CSV_PATH, index=False)

print(f"Success! Data saved to {CLEAN_DATA_CSV_PATH}")

C:\Users\obaid\AppData\Local\Temp\ipykernel_12012\3537980933.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_clean = pd.read_sql_query(query, conn)


Success! Data saved to 25280049_clean_synthetic_dataset.csv


In [19]:
# Store Corrupted Data

query = f"""
SELECT * FROM {CORRUPTED_DATA_TABLE_NAME};
"""

df_corrupted = pd.read_sql_query(query, conn)
df_corrupted.to_csv(CORRUPTED_DATA_CSV_PATH, index=False)

print(f"Success! Data saved to {CORRUPTED_DATA_CSV_PATH}")

C:\Users\obaid\AppData\Local\Temp\ipykernel_12012\3988984928.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_corrupted = pd.read_sql_query(query, conn)


Success! Data saved to 25280049_corrupted_synthetic_dataset.csv


## Task 2: Data Quality Report with Great Expectations

In [20]:
# Set up Great Expectations on an in-memory pandas DataFrame.
def build_batch_from_dataframe(df: pd.DataFrame):
    """
    Create an ephemeral GX context and connect the pandas DataFrame as a Batch.
    This mirrors the current GX Core workflow for dataframe data.

    Remarks: In the ephemeral GX context of Great Expectations, ephemeral means temporary and in-memory, 
             without being saved as a persistent project or configuration on disk. In an ephemeral 
             workflow, validations are created and executed programmatically within a script or notebook, 
             often on a pandas DataFrame, and the configuration (datasource, expectations, validator) 
             exists only for that session. In short, ephemeral GX is used for quick, lightweight 
             validations during exploration, testing, or pipelines, where you do not need a full 
             persisted GX project structure.
    """
    # Creates a Great Expectations context. 
    # A context is the main object that manages:
    #                                           - datasources
    #                                           - data assets
    #                                           - validation configurations
    #                                           - expectations
    # Since no project folder is loaded, this becomes an ephemeral context (temporary and in-memory).
    context             = gx.get_context()

    # Registers a pandas datasource inside GX. A datasource tells GX where the data comes from.
    data_source         = context.data_sources.add_pandas(name = "assignment3_pandas_source")

    # Defines a data asset. A data asset represents a logical dataset inside the datasource.
    # Example meanings of assets:
    #                           - orders
    #                           - transactions
    #                           - customer_records
    # Here the asset represents the orders DataFrame.
    data_asset          = data_source.add_dataframe_asset(name = "orders_dataframe_asset")
    
    # Defines how batches of data will be created. A Batch in GX is: the specific slice of data 
    # that will be validated.
    batch_definition    = data_asset.add_batch_definition_whole_dataframe("whole_orders_dataframe")
    
    # This step attaches the actual pandas DataFrame to the GX batch.
    # GX now knows:
    #               - which dataframe to validate,
    #               - which asset it belongs to,
    #               - which datasource it came from.
    batch               = batch_definition.get_batch(batch_parameters = {"dataframe": df})

    return context, batch_definition, batch

In [21]:
def build_expectations() -> List[Any]:
    expectations: List[Any] = [

        # Schema enforcement 
        gx.expectations.ExpectColumnToExist(column="sale_id",      severity="critical"),
        gx.expectations.ExpectColumnToExist(column="customer_id",  severity="critical"),
        gx.expectations.ExpectColumnToExist(column="product_id",   severity="critical"),

        # Data type enforcement 
        gx.expectations.ExpectColumnValuesToBeOfType(column="sale_id",     type_="int64",          severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="customer_id", type_="int64",          severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="product_id",  type_="int64",          severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="sale_ts",     type_="datetime64[ns]", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="quantity",    type_="int64",          severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="amount",      type_="float64",        severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="channel",     type_="str",         severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="currency",    type_="str",         severity="critical"),

        # Completeness / NOT NULL columns 
        gx.expectations.ExpectColumnValuesToNotBeNull(column="customer_id", severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="product_id",  severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_ts",     severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="quantity",    severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="amount",      severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="channel",     severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="currency",    severity="critical"),

        # notes is nullable
        gx.expectations.ExpectColumnValuesToNotBeNull(column="notes", mostly=0.0, severity="warning"),

        # Uniqueness 
        gx.expectations.ExpectColumnValuesToBeUnique(column="sale_id", severity="critical"),

        # Controlled vocabulary 
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="channel",
            value_set=["online", "mobile_app", "store", "partner"],
            severity="critical",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="currency",
            value_set=["PKR", "USD", "GBP", "EUR"],
            severity="critical",
        ),

        # Business-rule validation
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="amount",
            min_value=0,
            strict_min=True, 
            severity="critical"
        )
    ]

    return expectations

In [22]:
def gx_result_to_dict(result: Any) -> Dict[str, Any]:
    """
    This function converts a validation result object (often returned by Great Expectations) into a 
    standard Python dictionary so it can be easily serialized, logged, saved to JSON, or reported. 
    It is designed to be robust and compatible with multiple object formats.
    """
    # Checks whether the result is already a dictionary. If Yes, no conversion needed.
    if isinstance(result, dict): return result
    
    # The function checks if the object has any of these methods.
    for method_name in ("to_json_dict", "as_dict", "model_dump"):
        method  = getattr(result, method_name, None)
        if callable(method):
            try:
                converted = method()
                if isinstance(converted, dict):
                    return converted
            except Exception:
                pass
    
    # Try JSON string methods
    for method_name in ("to_json", "model_dump_json", "json"):
        method  = getattr(result, method_name, None)
        if callable(method):
            try:
                # Convert JSON string into dictionary
                raw = method()
                if isinstance(raw, str):
                    return json.loads(raw)
            except Exception:
                pass

    # Try direct dictionary casting            
    try:
        return dict(result)
    except Exception:
        # Last fallback - if everything fails - This stores the string representation of the object.
        return {"repr": repr(result)}

In [23]:
def summarize_single_expectation_result(result_dict: Dict[str, Any]) -> Dict[str, Any]:
    """
    This function extracts a small, readable summary from a single validation result produced by 
    Great Expectations. Instead of returning the full verbose validation output, it keeps only the most 
    useful fields for reporting or dashboards.
    """
    # Extract configuration information.
    config  = result_dict.get("expectation_config", {})

    # This retrieves the actual validation output produced by GX.
    res     = result_dict.get("result", {})

    # The function returns a new dictionary containing selected fields:
    return {"expectation_type"          : config.get("type") or config.get("expectation_type", "unknown"),
            "success"                   : result_dict.get("success"),
            "unexpected_count"          : res.get("unexpected_count"),
            "unexpected_percent"        : res.get("unexpected_percent"),
            "partial_unexpected_list"   : res.get("partial_unexpected_list", []),}

In [24]:
# Main execution flow for Great Expectations validation on the DataFrame.
def main(df: pd.DataFrame, summary_csv_path: str, suite_name: str, order_validation_def: str, suite_result_path: str, output_dir: str):

    os.makedirs(output_dir, exist_ok=True)

    # Create GX context, datasource, asset, batch definition, and batch
    context, batch_definition, batch = build_batch_from_dataframe(df)
    
    print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
    print(f"Context type: {type(context).__name__}")
    print("Batch created successfully.")

    # Define expectations
    expectations    = build_expectations()
    print(f"Number of expectations defined: {len(expectations)}")

    for idx, expectation in enumerate(expectations, start=1):
        print(f"{idx:02d}. {type(expectation).__name__}")

    # Run ad hoc validations one expectation at a time
    summaries: List[Dict[str, Any]] = []
    for expectation in expectations:
        result      = batch.validate(expectation)
        result_dict = gx_result_to_dict(result)
        summary     = summarize_single_expectation_result(result_dict)
        summaries.append(summary)

        print(f"\nExpectation   : {summary['expectation_type']}")
        print(f"Success         : {summary['success']}")
        print(f"Unexpected      : {summary['unexpected_count']}")
        
        if summary["partial_unexpected_list"]:
            print(f"Examples    : {summary['partial_unexpected_list']}")

    summary_df          = pd.DataFrame(summaries)
    summary_csv_path    = output_dir / summary_csv_path
    summary_df.to_csv(summary_csv_path, index=False)

    print("\nCompact validation summary:")
    print(summary_df)
    print(f"\nSaved summary to: {summary_csv_path.resolve()}")

    # Organize expectations into an Expectation Suite
    suite           = gx.ExpectationSuite(name=suite_name)
    suite           = context.suites.add(suite)

    for expectation in expectations: suite.add_expectation(expectation)

    print(f"Expectation Suite name  : {suite.name}")
    print(f"Expectations in suite   :  {len(suite.expectations)}")

    # Create a Validation Definition and validate the whole suite
    validation_definition = context.validation_definitions.add(
        gx.core.validation_definition.ValidationDefinition(name = order_validation_def,
                                                           data = batch_definition,
                                                           suite= suite,))

    suite_result        = validation_definition.run(batch_parameters={"dataframe": df})
    suite_result_path   = output_dir / suite_result_path
    suite_result_dict   = gx_result_to_dict(suite_result)

    with suite_result_path.open("w", encoding="utf-8") as f:
        json.dump(suite_result_dict, f, indent=2, default=str)

    print("Full suite validation finished.")
    print(f"Saved suite result JSON to: {suite_result_path.resolve()}")
    

#### Great Expectation on Clean Dataset

In [25]:
main(df = df_clean,
     summary_csv_path = Path("25280049_validation_summary_clean.csv"),
     suite_name = "25280049_clean_data_suite",
     order_validation_def = "25280049_clean_data_validation_definition",
     suite_result_path = Path("25280049_clean_data_suite_result.json"),
     output_dir = Path("25280049_clean_data_great_expectations_results")
)

GX version: 1.15.2
Context type: EphemeralDataContext
Batch created successfully.
Number of expectations defined: 23
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnValuesToBeOfType
05. ExpectColumnValuesToBeOfType
06. ExpectColumnValuesToBeOfType
07. ExpectColumnValuesToBeOfType
08. ExpectColumnValuesToBeOfType
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToNotBeNull
13. ExpectColumnValuesToNotBeNull
14. ExpectColumnValuesToNotBeNull
15. ExpectColumnValuesToNotBeNull
16. ExpectColumnValuesToNotBeNull
17. ExpectColumnValuesToNotBeNull
18. ExpectColumnValuesToNotBeNull
19. ExpectColumnValuesToNotBeNull
20. ExpectColumnValuesToBeUnique
21. ExpectColumnValuesToBeInSet
22. ExpectColumnValuesToBeInSet
23. ExpectColumnValuesToBeBetween


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 874.09it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 872.45it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


...


Expectation   : expect_column_values_to_be_in_set
Success         : True
Unexpected      : 0


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 899.27it/s] 



Expectation   : expect_column_values_to_be_between
Success         : True
Unexpected      : 0

Compact validation summary:
                       expectation_type  success  unexpected_count  \
0                expect_column_to_exist     True               NaN   
1                expect_column_to_exist     True               NaN   
2                expect_column_to_exist     True               NaN   
3    expect_column_values_to_be_of_type     True               NaN   
4    expect_column_values_to_be_of_type     True               NaN   
5    expect_column_values_to_be_of_type     True               NaN   
6    expect_column_values_to_be_of_type     True               NaN   
7    expect_column_values_to_be_of_type     True               NaN   
8    expect_column_values_to_be_of_type     True               NaN   
9    expect_column_values_to_be_of_type     True               NaN   
10   expect_column_values_to_be_of_type     True               NaN   
11  expect_column_values_to_not_be_n

Calculating Metrics: 100%|██████████| 72/72 [00:00<00:00, 390.95it/s]


Full suite validation finished.
Saved suite result JSON to: D:\LUMS\Semester_2\Fundamentals_of_Data_Engineering\Assignments\3\25280049_clean_data_great_expectations_results\25280049_clean_data_suite_result.json


#### Great Expectation on Corrupted Dataset

In [26]:
main(df = df_corrupted,
     summary_csv_path = Path("25280049_validation_summary_corrupted.csv"),
     suite_name = "25280049_corrupted_data_suite",
     order_validation_def = "25280049_corrupted_data_validation_definition",
     suite_result_path = Path("25280049_corrupted_data_suite_result.json"),
     output_dir = Path("25280049_corrupted_data_great_expectations_results")
)

GX version: 1.15.2
Context type: EphemeralDataContext
Batch created successfully.
Number of expectations defined: 23
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnValuesToBeOfType
05. ExpectColumnValuesToBeOfType
06. ExpectColumnValuesToBeOfType
07. ExpectColumnValuesToBeOfType
08. ExpectColumnValuesToBeOfType
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToNotBeNull
13. ExpectColumnValuesToNotBeNull
14. ExpectColumnValuesToNotBeNull
15. ExpectColumnValuesToNotBeNull
16. ExpectColumnValuesToNotBeNull
17. ExpectColumnValuesToNotBeNull
18. ExpectColumnValuesToNotBeNull
19. ExpectColumnValuesToNotBeNull
20. ExpectColumnValuesToBeUnique
21. ExpectColumnValuesToBeInSet
22. ExpectColumnValuesToBeInSet
23. ExpectColumnValuesToBeBetween


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 1071.62it/s]



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 824.92it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


...


Expectation   : expect_column_values_to_be_in_set
Success         : False
Unexpected      : 37620
Examples    : ['$138.91', '57,93', '64.36 GBP', '$18.86', '$39.70', '195,03', '171,87', '$112.18', '21,42', '$127.19', '82,67', '104,61', '$82.22', '$71.52', '81.11 GBP', '$427.75', '31,02', '58.21 GBP', '75.15 EUR', '$98.27']


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 466.25it/s]



Expectation   : expect_column_values_to_be_between
Success         : False
Unexpected      : 19634
Examples    : [0.0, 0.0, 0.0, 0.0, 0.0, -369.19, 0.0, 0.0, 0.0, 0.0, -94.53, 0.0, 0.0, 0.0, 0.0, 0.0, -185.86, -423.45, 0.0, 0.0]

Compact validation summary:
                       expectation_type  success  unexpected_count  \
0                expect_column_to_exist     True               NaN   
1                expect_column_to_exist     True               NaN   
2                expect_column_to_exist     True               NaN   
3    expect_column_values_to_be_of_type     True               NaN   
4    expect_column_values_to_be_of_type    False               NaN   
5    expect_column_values_to_be_of_type    False               NaN   
6    expect_column_values_to_be_of_type     True               NaN   
7    expect_column_values_to_be_of_type     True               NaN   
8    expect_column_values_to_be_of_type     True               NaN   
9    expect_column_values_to_be_of_type  

Calculating Metrics: 100%|██████████| 72/72 [00:00<00:00, 97.41it/s] 

Full suite validation finished.
Saved suite result JSON to: D:\LUMS\Semester_2\Fundamentals_of_Data_Engineering\Assignments\3\25280049_corrupted_data_great_expectations_results\25280049_corrupted_data_suite_result.json


## Task 3: Data Quality Report with Great Expectations on Pak-Wheels Dataset

#### Load Pak-Wheels Dataset

In [27]:
df_pakwheels = pd.read_csv("data-a2.csv")
df_pakwheels.head()

,addref,city,assembly,body,make,model,year,engine,transmission,fuel,color,registered,mileage,price
0,7642697,Islamabad,NaN,Compact SUV,KIA,Sorento,2021.0,3500.0,Automatic,Petrol,NaN,Islamabad,54654,9300000.0
1,7909608,Sadiqabad,Imported,Hatchback,Daihatsu,Mira,2020.0,660.0,Automatic,Petrol,White,Punjab,10000,3700000.0
2,7929224,Peshawar,Imported,Hatchback,Toyota,Vitz,2018.0,1000.0,Automatic,Petrol,Silver,Islamabad,123000,4150000.0
3,7832366,Lahore,NaN,Sedan,Toyota,Corolla,2019.0,1600.0,Automatic,Petrol,White,Lahore,105000,4850000.0
4,7866725,Islamabad,NaN,Sedan,Toyota,Corolla,2022.0,1600.0,Automatic,Petrol,White,Islamabad,6500,6600000.0


#### Build Expectation Pakwheels

In [28]:
def build_expectations_pakwheels() -> List[Any]:
    """
    Expectations for the PakWheels used-car listings table.
    Critical = pipeline-blocking; warning = logged but non-blocking.
    """
    expectations: List[Any] = [

        #  Schema enforcement 
        gx.expectations.ExpectColumnToExist(column="addref",        severity="critical"),
        gx.expectations.ExpectColumnToExist(column="city",          severity="critical"),
        gx.expectations.ExpectColumnToExist(column="make",          severity="critical"),
        gx.expectations.ExpectColumnToExist(column="model",         severity="critical"),
        gx.expectations.ExpectColumnToExist(column="year",          severity="critical"),
        gx.expectations.ExpectColumnToExist(column="price",         severity="critical"),

        #  Data type enforcement 
        gx.expectations.ExpectColumnValuesToBeOfType(column="addref",       type_="int64",   severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="year",         type_="float64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="engine",       type_="float64", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="mileage",      type_="float64", severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="price",        type_="float64", severity="critical"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="city",         type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="assembly",     type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="body",         type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="make",         type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="model",        type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="transmission", type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="fuel",         type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="color",        type_="str",  severity="warning"),
        gx.expectations.ExpectColumnValuesToBeOfType(column="registered",   type_="str",  severity="warning"),

        #  Completeness 
        gx.expectations.ExpectColumnValuesToNotBeNull(column="addref",  severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="make",    severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="model",   severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="year",    severity="critical"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="price",   severity="critical"),

        # Frequently missing in real scrapes; warn, don't block.
        gx.expectations.ExpectColumnValuesToNotBeNull(column="city",         mostly=0.90, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="assembly",     mostly=0.70, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="body",         mostly=0.80, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="engine",       mostly=0.85, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="transmission", mostly=0.85, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="fuel",         mostly=0.85, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="color",        mostly=0.70, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="registered",   mostly=0.75, severity="warning"),
        gx.expectations.ExpectColumnValuesToNotBeNull(column="mileage",      mostly=0.80, severity="warning"),

        #  Uniqueness 
        gx.expectations.ExpectColumnValuesToBeUnique(column="addref", severity="critical"),

        #  Value-range checks 
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="year", min_value=1980, max_value=2026, severity="critical",
        ),
        # Engine cc
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="engine", min_value=600, max_value=6500, severity="warning",
        ),
        # Mileage
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="mileage", min_value=0, max_value=1_000_000, severity="warning",
        ),
        # Price in PKR
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="price", min_value=50_000, max_value=100_000_000, severity="critical",
        ),
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="year", min_value=1970, max_value=2026, severity="critical"
        ),
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="price", min_value=-0, strict_min=True, severity="critical"
        ),

        #  Controlled vocabulary 
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="assembly",
            value_set=["Local", "Imported"],
            severity="warning",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="transmission",
            value_set=["Automatic", "Manual"],
            severity="warning",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="fuel",
            value_set=["Petrol", "Diesel", "Hybrid"],
            severity="warning",
        ),
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="body",
            value_set=[
                "Compact hatchback", "Compact sedan", "SUV", "Compact SUV", "Crossover",
                "Micro Van", "High Roof", "Hatchback", "Double Cabin", "Coupe", "Convertible",
                "Mini Van", "Van", "Pick Up", "Mini Vehicales", "MPV", "Off-Road Vehicles", "Sedan", 
                "Single Cabin", "Station Wagon", "Truck"
            ],
            severity="warning",
        )
    ]

    return expectations

In [29]:
# Main execution flow for Great Expectations validation on the Pakwheels DataFrame.

def main_pakwheels(df: pd.DataFrame, summary_csv_path: str, suite_name: str, order_validation_def: str, suite_result_path: str, output_dir: str):
    
    os.makedirs(output_dir, exist_ok=True)

    # Create GX context, datasource, asset, batch definition, and batch
    context, batch_definition, batch = build_batch_from_dataframe(df)
    
    print(f"GX version: {getattr(gx, '__version__', 'unknown')}")
    print(f"Context type: {type(context).__name__}")
    print("Batch created successfully.")

    # Define expectations
    expectations    = build_expectations_pakwheels()
    print(f"Number of expectations defined: {len(expectations)}")

    for idx, expectation in enumerate(expectations, start=1):
        print(f"{idx:02d}. {type(expectation).__name__}")

    # Run ad hoc validations one expectation at a time
    summaries: List[Dict[str, Any]] = []
    for expectation in expectations:
        result      = batch.validate(expectation)
        result_dict = gx_result_to_dict(result)
        summary     = summarize_single_expectation_result(result_dict)
        summaries.append(summary)

        print(f"\nExpectation   : {summary['expectation_type']}")
        print(f"Success         : {summary['success']}")
        print(f"Unexpected      : {summary['unexpected_count']}")
        
        if summary["partial_unexpected_list"]:
            print(f"Examples    : {summary['partial_unexpected_list']}")

    summary_df          = pd.DataFrame(summaries)
    summary_csv_path    = output_dir / summary_csv_path
    summary_df.to_csv(summary_csv_path, index=False)

    print("\nCompact validation summary:")
    print(summary_df)
    print(f"\nSaved summary to: {summary_csv_path.resolve()}")

    # Organize expectations into an Expectation Suite
    suite           = gx.ExpectationSuite(name=suite_name)
    suite           = context.suites.add(suite)

    for expectation in expectations: suite.add_expectation(expectation)

    print(f"Expectation Suite name  : {suite.name}")
    print(f"Expectations in suite   :  {len(suite.expectations)}")

    # Create a Validation Definition and validate the whole suite
    validation_definition = context.validation_definitions.add(
        gx.core.validation_definition.ValidationDefinition(name = order_validation_def,
                                                           data = batch_definition,
                                                           suite= suite,))

    suite_result        = validation_definition.run(batch_parameters={"dataframe": df})
    suite_result_path   = output_dir / suite_result_path
    suite_result_dict   = gx_result_to_dict(suite_result)

    with suite_result_path.open("w", encoding="utf-8") as f:
        json.dump(suite_result_dict, f, indent=2, default=str)

    print("Full suite validation finished.")
    print(f"Saved suite result JSON to: {suite_result_path.resolve()}")
    

In [30]:
main_pakwheels(df = df_pakwheels,
     summary_csv_path = Path("25280049_validation_summary_pakwheels.csv"),
     suite_name = "25280049_pakwheels_data_suite",
     order_validation_def = "25280049_pakwheels_data_validation_definition",
     suite_result_path = Path("25280049_pakwheels_data_suite_result.json"),
     output_dir = Path("25280049_pakwheels_data_great_expectations_results")
)

GX version: 1.15.2
Context type: EphemeralDataContext
Batch created successfully.
Number of expectations defined: 45
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnToExist
05. ExpectColumnToExist
06. ExpectColumnToExist
07. ExpectColumnValuesToBeOfType
08. ExpectColumnValuesToBeOfType
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToBeOfType
13. ExpectColumnValuesToBeOfType
14. ExpectColumnValuesToBeOfType
15. ExpectColumnValuesToBeOfType
16. ExpectColumnValuesToBeOfType
17. ExpectColumnValuesToBeOfType
18. ExpectColumnValuesToBeOfType
19. ExpectColumnValuesToBeOfType
20. ExpectColumnValuesToBeOfType
21. ExpectColumnValuesToNotBeNull
22. ExpectColumnValuesToNotBeNull
23. ExpectColumnValuesToNotBeNull
24. ExpectColumnValuesToNotBeNull
25. ExpectColumnValuesToNotBeNull
26. ExpectColumnValuesToNotBeNull
27. ExpectColumnValuesToNotBeNull
28. ExpectColumnValuesToNotBeNull
29. E

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 806.05it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 687.87it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


...


Expectation   : expect_column_values_to_be_in_set
Success         : True
Unexpected      : 0


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 265.47it/s]



Expectation   : expect_column_values_to_be_in_set
Success         : False
Unexpected      : 5
Examples    : ['Mini Vehicles', 'Mini Vehicles', 'Mini Vehicles', 'Mini Vehicles', 'Mini Vehicles']

Compact validation summary:
                       expectation_type  success  unexpected_count  \
0                expect_column_to_exist     True               NaN   
1                expect_column_to_exist     True               NaN   
2                expect_column_to_exist     True               NaN   
3                expect_column_to_exist     True               NaN   
4                expect_column_to_exist     True               NaN   
5                expect_column_to_exist     True               NaN   
6    expect_column_values_to_be_of_type     True               NaN   
7    expect_column_values_to_be_of_type     True               NaN   
8    expect_column_values_to_be_of_type     True               NaN   
9    expect_column_values_to_be_of_type    False               NaN   
10   e

Calculating Metrics: 100%|██████████| 139/139 [00:00<00:00, 486.80it/s]

Full suite validation finished.
Saved suite result JSON to: D:\LUMS\Semester_2\Fundamentals_of_Data_Engineering\Assignments\3\25280049_pakwheels_data_great_expectations_results\25280049_pakwheels_data_suite_result.json


## Task 4: Data Cleaning and Re-validation

### Data Cleaning of Corrupted Data 

In [31]:
# Based on The Great Expectations Results Corrupted Data has 10 errors
# 2 in Data Types (customer_id and product_id have text values)
# 4 in Nullability (customer_id, product_id, sale_ts, channel)
# 1 in Uniqueness (sale_id has duplicates)
# 3 in Value Ranges (channel, currency, amount columns have invalid values) 

df_corrupted_fix = df_corrupted.copy()

# drop duplicates based on sale_id
df_corrupted_fix = df_corrupted_fix.drop_duplicates(subset=["sale_id"], keep="first")

# change cusotmer_id and product_id to numeric
df_corrupted_fix["customer_id"] = pd.to_numeric(df_corrupted_fix["customer_id"], errors="coerce").astype("Int64")
df_corrupted_fix["product_id"] = pd.to_numeric(df_corrupted_fix["product_id"], errors="coerce").astype("Int64")

# drop tables where no customer_id or product_id
df_corrupted_fix = df_corrupted_fix.dropna(subset=["customer_id", "product_id"])
# fill missing values in channel with 'unknown'
df_corrupted_fix["channel"] = df_corrupted_fix["channel"].fillna("unknown")
# fill missing values in sale_ts with a placeholder date
df_corrupted_fix["sale_ts"] = pd.to_datetime(df_corrupted_fix["sale_ts"], errors="coerce").fillna(pd.Timestamp("1970-01-01"))

# fix invalid channels by replacing non-conforming values with 'unknown'
valid_channels = ["online", "mobile_app", "store", "partner"]
df_corrupted_fix = df_corrupted_fix[df_corrupted_fix["channel"].isin(valid_channels)]
# fix invalid currencies by replacing non-conforming values with 'unknown'
valid_currencies = ["PKR", "USD", "GBP", "EUR"]
df_corrupted_fix = df_corrupted_fix[df_corrupted_fix["currency"].isin(valid_currencies)]

# fix invalid amounts by setting negative or zero values to NaN and then filling with median
def clean_amount(val):
    if pd.isna(val): 
        return np.nan
    
    val = str(val).replace(',', '.') 

    cleaned = "".join([char for char in val if char.isdigit() or char in "."])

    try:
        return float(cleaned)
    except ValueError:
        return np.nan
    
df_corrupted_fix["amount"] = df_corrupted_fix["amount"].apply(clean_amount)
df_corrupted_fix["amount"] = df_corrupted_fix["amount"].where(df_corrupted_fix["amount"] > 0, other=np.nan)
# fill empty with median
df_corrupted_fix["amount"] = df_corrupted_fix["amount"].fillna(df_corrupted_fix["amount"].median())

# Re-run Great Expectations validation on the cleaned DataFrame to confirm all issues are resolved.
main(df = df_corrupted_fix,
        summary_csv_path = Path("25280049_validation_summary_corrupted_after_fix.csv"),
        suite_name = "25280049_corrupted_data_suite_after_fix",
        order_validation_def = "25280049_corrupted_data_validation_definition_after_fix",
        suite_result_path = Path("25280049_corrupted_data_suite_result_after_fix.json"),
        output_dir = Path("25280049_corrupted_data_great_expectations_results_after_fix")
    )

df_corrupted_fix.to_csv("25280049_corrupted_data_fixed.csv", index=False)

GX version: 1.15.2
Context type: EphemeralDataContext
Batch created successfully.
Number of expectations defined: 23
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnValuesToBeOfType
05. ExpectColumnValuesToBeOfType
06. ExpectColumnValuesToBeOfType
07. ExpectColumnValuesToBeOfType
08. ExpectColumnValuesToBeOfType
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToNotBeNull
13. ExpectColumnValuesToNotBeNull
14. ExpectColumnValuesToNotBeNull
15. ExpectColumnValuesToNotBeNull
16. ExpectColumnValuesToNotBeNull
17. ExpectColumnValuesToNotBeNull
18. ExpectColumnValuesToNotBeNull
19. ExpectColumnValuesToNotBeNull
20. ExpectColumnValuesToBeUnique
21. ExpectColumnValuesToBeInSet
22. ExpectColumnValuesToBeInSet
23. ExpectColumnValuesToBeBetween


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 1002.46it/s]



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 1017.42it/s]



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


...


Expectation   : expect_column_values_to_be_in_set
Success         : True
Unexpected      : 0


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 1107.61it/s]



Expectation   : expect_column_values_to_be_between
Success         : True
Unexpected      : 0

Compact validation summary:
                       expectation_type  success  unexpected_count  \
0                expect_column_to_exist     True               NaN   
1                expect_column_to_exist     True               NaN   
2                expect_column_to_exist     True               NaN   
3    expect_column_values_to_be_of_type     True               NaN   
4    expect_column_values_to_be_of_type     True               NaN   
5    expect_column_values_to_be_of_type     True               NaN   
6    expect_column_values_to_be_of_type     True               NaN   
7    expect_column_values_to_be_of_type     True               NaN   
8    expect_column_values_to_be_of_type     True               NaN   
9    expect_column_values_to_be_of_type     True               NaN   
10   expect_column_values_to_be_of_type     True               NaN   
11  expect_column_values_to_not_be_n

Calculating Metrics: 100%|██████████| 72/72 [00:00<00:00, 280.77it/s]


Full suite validation finished.
Saved suite result JSON to: D:\LUMS\Semester_2\Fundamentals_of_Data_Engineering\Assignments\3\25280049_corrupted_data_great_expectations_results_after_fix\25280049_corrupted_data_suite_result_after_fix.json


### Data Cleaning of Pakwheels Data 

In [32]:
# Based on The Great Expectations Results Corrupted Data has 18 errors
# 10 in Data Types 
# 3 in Nullability 
# 2 in Value Between a Range 
# 3 in Value Sets 

df_pakwheels_fix = df_pakwheels.copy()


# change mileage to numeric
df_pakwheels_fix["mileage"] = pd.to_numeric(df_pakwheels_fix["mileage"], errors="coerce").astype("Float64")

# fill rest with mode
df_pakwheels_fix["city"] = df_pakwheels_fix["city"].fillna(df_pakwheels_fix["city"].mode()[0])
df_pakwheels_fix["assembly"] = df_pakwheels_fix["assembly"].fillna(df_pakwheels_fix["assembly"].mode()[0])
df_pakwheels_fix["body"] = df_pakwheels_fix["body"].fillna(df_pakwheels_fix["body"].mode()[0])
df_pakwheels_fix["make"] = df_pakwheels_fix["make"].fillna(df_pakwheels_fix["make"].mode()[0])
df_pakwheels_fix["model"] = df_pakwheels_fix["model"].fillna(df_pakwheels_fix["model"].mode()[0])
df_pakwheels_fix["transmission"] = df_pakwheels_fix["transmission"].fillna(df_pakwheels_fix["transmission"].mode()[0])
df_pakwheels_fix["fuel"] = df_pakwheels_fix["fuel"].fillna(df_pakwheels_fix["fuel"].mode()[0])
df_pakwheels_fix["color"] = df_pakwheels_fix["color"].fillna(df_pakwheels_fix["color"].mode()[0])
df_pakwheels_fix["registered"] = df_pakwheels_fix["registered"].fillna(df_pakwheels_fix["registered"].mode()[0])

# fill null values
df_pakwheels_fix["year"] = df_pakwheels_fix["year"].fillna(df_pakwheels_fix["year"].median())
df_pakwheels_fix["price"] = df_pakwheels_fix["price"].fillna(df_pakwheels_fix["price"].median())
df_pakwheels_fix["assembly"] = df_pakwheels_fix["assembly"].fillna(df_pakwheels_fix["assembly"].mode()[0])

# drop rows where invalid ranges of engine and price
df_pakwheels_fix = df_pakwheels_fix[(df_pakwheels_fix["engine"].between(600, 6500)) & 
                                    (df_pakwheels_fix["price"].between(50_000, 100_000_000))]

# if body not in value set, replace with mode
valid_bodies = [
                "Compact hatchback", "Compact sedan", "SUV", "Compact SUV", "Crossover",
                "Micro Van", "High Roof", "Hatchback", "Double Cabin", "Coupe", "Convertible",
                "Mini Van", "Van", "Pick Up", "Mini Vehicales", "MPV", "Off-Road Vehicles", "Sedan", 
                "Single Cabin", "Station Wagon", "Truck"
            ]
df_pakwheels_fix["body"] = df_pakwheels_fix["body"].where(df_pakwheels_fix["body"].isin(valid_bodies), other=df_pakwheels_fix["body"].mode()[0])

# for value sets, replace invalid values with mode

# Re-run Great Expectations validation on the cleaned DataFrame to confirm all issues are resolved.
main_pakwheels(df = df_pakwheels_fix,
        summary_csv_path = Path("25280049_validation_summary_pakwheels_after_fix.csv"),
        suite_name = "25280049_pakwheels_data_suite_after_fix",
        order_validation_def = "25280049_pakwheels_data_validation_definition_after_fix",
        suite_result_path = Path("25280049_pakwheels_data_suite_result_after_fix.json"),
        output_dir = Path("25280049_pakwheels_data_great_expectations_results_after_fix")
    )

df_pakwheels_fix.to_csv("25280049_pakwheels_data_fixed.csv", index=False)

GX version: 1.15.2
Context type: EphemeralDataContext
Batch created successfully.
Number of expectations defined: 45
01. ExpectColumnToExist
02. ExpectColumnToExist
03. ExpectColumnToExist
04. ExpectColumnToExist
05. ExpectColumnToExist
06. ExpectColumnToExist
07. ExpectColumnValuesToBeOfType
08. ExpectColumnValuesToBeOfType
09. ExpectColumnValuesToBeOfType
10. ExpectColumnValuesToBeOfType
11. ExpectColumnValuesToBeOfType
12. ExpectColumnValuesToBeOfType
13. ExpectColumnValuesToBeOfType
14. ExpectColumnValuesToBeOfType
15. ExpectColumnValuesToBeOfType
16. ExpectColumnValuesToBeOfType
17. ExpectColumnValuesToBeOfType
18. ExpectColumnValuesToBeOfType
19. ExpectColumnValuesToBeOfType
20. ExpectColumnValuesToBeOfType
21. ExpectColumnValuesToNotBeNull
22. ExpectColumnValuesToNotBeNull
23. ExpectColumnValuesToNotBeNull
24. ExpectColumnValuesToNotBeNull
25. ExpectColumnValuesToNotBeNull
26. ExpectColumnValuesToNotBeNull
27. ExpectColumnValuesToNotBeNull
28. ExpectColumnValuesToNotBeNull
29. E

Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 805.28it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


Calculating Metrics: 100%|██████████| 2/2 [00:00<00:00, 697.42it/s] 



Expectation   : expect_column_to_exist
Success         : True
Unexpected      : None


...


Expectation   : expect_column_values_to_be_in_set
Success         : True
Unexpected      : 0


Calculating Metrics: 100%|██████████| 10/10 [00:00<00:00, 1155.65it/s]



Expectation   : expect_column_values_to_be_in_set
Success         : True
Unexpected      : 0

Compact validation summary:
                       expectation_type  success  unexpected_count  \
0                expect_column_to_exist     True               NaN   
1                expect_column_to_exist     True               NaN   
2                expect_column_to_exist     True               NaN   
3                expect_column_to_exist     True               NaN   
4                expect_column_to_exist     True               NaN   
5                expect_column_to_exist     True               NaN   
6    expect_column_values_to_be_of_type     True               NaN   
7    expect_column_values_to_be_of_type     True               NaN   
8    expect_column_values_to_be_of_type     True               NaN   
9    expect_column_values_to_be_of_type     True               NaN   
10   expect_column_values_to_be_of_type     True               NaN   
11   expect_column_values_to_be_of_ty

Calculating Metrics: 100%|██████████| 139/139 [00:00<00:00, 2132.49it/s]


Full suite validation finished.
Saved suite result JSON to: D:\LUMS\Semester_2\Fundamentals_of_Data_Engineering\Assignments\3\25280049_pakwheels_data_great_expectations_results_after_fix\25280049_pakwheels_data_suite_result_after_fix.json


AI Usage: 

1. Some of the code in this was auto suggested by copilot where the code logic was to be repeated for multiple tables. Initial logic was manually implemented, then if repeated the co-pilot used to fill in.
2. Used help from GPT to idently what expectation to be created for each column and used that with a few modifications made by me.